# quantum-cortex — N2 MQAR ablation v2 (delta stabilised + AUC verdict)

v1 learnings: Hopfield beats control on AUC (+14%, masked by the 0.5 threshold); delta v1 diverged on all tiers (fixed: L2-normalised keys). This re-runs the three variants with the fix, reads the verdict by **AUC** (whole-curve), and optionally sweeps Hopfield capacity (dimensioning vs structure).

GPU T4, Internet On.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema

In [ ]:
# Full ablation v2: delta is now stable (L2-normalised keys). Verdict by AUC.
!python -m cortex_eval.ablation_mqar --steps 1500 --out metrics/mqar

In [ ]:
# Read the AUC verdict and the three curves
import json
r = json.load(open('metrics/mqar/LATEST.json'))
print('VERDICT:', r['verdict'])
print('AUC:', json.dumps(r.get('auc',{}), indent=2))
print('AUC vs control:', json.dumps(r.get('auc_vs_control',{}), indent=2))
for v,c in r['curves'].items():
    print(f'--- {v} ---')
    for pt in c:
        a=pt['accuracy']; print(f"  kv={pt['kv_pairs']:>2} seq={pt['seq_len']:>4}  acc={'ABORT' if a is None else f'{a:.3f}'}")

## Optional: Hopfield capacity sweep (the founder's question)
Does Hopfield's limit come from memory *size* or *structure*? Run the cell below.

In [ ]:
from cortex_eval.ablation_mqar import run_hopfield_capacity_sweep
import json
sweep = run_hopfield_capacity_sweep(steps=1500, slots=(32,64,128,256))
print('READING:', sweep['reading'])
print(json.dumps(sweep['points'], indent=2))
json.dump(sweep, open('metrics/mqar/hopfield-capacity-sweep.json','w'), indent=2)